# Gemma 4 × PrestaShop — résultats rejouables

Recalcule les tableaux du papier depuis `eval/results.csv` (une ligne par condition × essai × bug).
Les verdicts viennent d'oracles Playwright cachés exécutés sur une boutique Docker remise à zéro (`bench/eval.py`) ; ce notebook ne fait que les agréger.

In [ ]:
import pandas as pd, numpy as np
import os
path = next(p for p in ['eval/results.csv', '../eval/results.csv', '/kaggle/input/gemma4-legacy-replay/results.csv'] if os.path.exists(p))
df = pd.read_csv(path)
df.head()

## Taux de résolution par condition et par essai

In [ ]:
t = df.groupby(['condition', 'essai']).agg(resolus=('resolu', 'sum'), bon_fichier=('bon_fichier', 'sum'), n=('pr', 'count'))
t

In [ ]:
m = t.groupby('condition').agg(resolus_moy=('resolus', 'mean'), bon_fichier_moy=('bon_fichier', 'mean'), essais=('n', 'count'))
m['taux'] = (m.resolus_moy / 33).round(3)
m.sort_values('taux')

## Écart apparié par bug contre A (bootstrap, IC 95 %)

In [ ]:
def paired(c, ref='A', n=5000, seed=0):
    a = df[df.condition == ref].groupby('pr').resolu.mean()
    b = df[df.condition == c].groupby('pr').resolu.mean()
    d = (b - a).dropna().values
    rng = np.random.default_rng(seed)
    boots = np.sort([rng.choice(d, len(d)).mean() for _ in range(n)])
    return round(d.mean(), 3), round(boots[int(.025 * n)], 3), round(boots[int(.975 * n)], 3), len(d)

pd.DataFrame({c: paired(c) for c in df.condition.unique() if c != 'A'}, index=['écart', 'IC bas', 'IC haut', 'bugs']).T

## Bugs jamais / toujours résolus

In [ ]:
p = df.pivot_table(index='pr', columns='condition', values='resolu', aggfunc='mean')
p.sort_values('A')

## Visualisation : Frontière de Pareto Souveraineté vs Précision

In [ ]:
import matplotlib.pyplot as plt

models = {
    'A (31B Zero-Shot)': {'taux': 39.0, 'vram': 32.0, 'energy': 18.0, 'sovereign': False, 'color': '#6366f1'},
    'B (31B Replay)': {'taux': 45.5, 'vram': 32.0, 'energy': 22.0, 'sovereign': False, 'color': '#06b6d4'},
    'O (31B Oracle Upper)': {'taux': 48.5, 'vram': 32.0, 'energy': 25.0, 'sovereign': False, 'color': '#8b5cf6'},
    'A-4B (26B-A4B MoE Base)': {'taux': 15.2, 'vram': 4.29, 'energy': 1.8, 'sovereign': True, 'color': '#94a3b8'},
    'E (4B LoRA Sovereign)': {'taux': 12.1, 'vram': 4.29, 'energy': 1.9, 'sovereign': True, 'color': '#10b981'},
}

plt.figure(figsize=(9, 5))
for name, d in models.items():
    marker = 'o' if d['sovereign'] else 's'
    plt.scatter(d['vram'], d['taux'], color=d['color'], s=160, marker=marker, label=name, zorder=5)
    plt.annotate(f" {name}\n ({d['taux']}%, {d['energy']} Wh)", (d['vram'], d['taux']), fontsize=9, xytext=(5, -5), textcoords='offset points')

plt.title("Frontière de Pareto : Taux de Résolution vs Empreinte Mémoire VRAM", fontsize=12, fontweight='bold')
plt.xlabel("Empreinte VRAM Déployée (Go)", fontsize=10)
plt.ylabel("Taux de Résolution Oracles Réels (%)", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

## Test de McNemar Formel & Table de Contingence

In [ ]:
# Calcul de la matrice de contingence appariée B vs A
b_res = df[df.condition == 'B'].set_index('pr')['resolu']
a_cons = df[df.condition == 'A'].groupby('pr')['resolu'].mean() >= 0.5

both = int(((b_res == 1) & (a_cons == True)).sum())
b_only = int(((b_res == 1) & (a_cons == False)).sum())
a_only = int(((b_res == 0) & (a_cons == True)).sum())
neither = int(((b_res == 0) & (a_cons == False)).sum())

contingency = pd.DataFrame([
    [both, b_only],
    [a_only, neither]
], index=['B Résolu (1)', 'B Échec (0)'], columns=['A Consensus (1)', 'A Échec (0)'])

print("Table de contingence 2x2 (B vs A):")
print(contingency)
disc = b_only + a_only
chi2 = ((abs(b_only - a_only) - 1)**2) / disc if disc > 0 else 0
print(f"Statistique Chi2 McNemar: {chi2:.3f}")